In [396]:

import pandas as pd
import numpy as np
from pathlib import Path

# --- LEITURA DOS ARQUIVOS DA ONS ---

# Caminho da pasta que contém os arquivos CSV
pasta_dados = Path("../dados/ONS")

# Localizar os arquivos de 2000 a 2025
arquivos = sorted(pasta_dados.glob("CARGA_ENERGIA_*.csv"))

print(f"Arquivos encontrados: {len(arquivos)}")

if len(arquivos) != 26:
    raise ValueError(
        f"Esperados 26 arquivos (2000 a 2025), mas foram encontrados {len(arquivos)}."
    )

# Ler, converter as datas individualmente e juntar os arquivos
lista_dfs = []

for arquivo in arquivos:
    df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8")

    # Converter as datas antes da concatenação
    df_temp["din_instante"] = pd.to_datetime(
        df_temp["din_instante"],
        errors="raise"
    )

    lista_dfs.append(df_temp)

df_ons = pd.concat(lista_dfs, ignore_index=True)

# Padronizar o nome da coluna de data
df_ons = df_ons.rename(columns={"din_instante": "data_hora"})

# Exibir informações da base
print("\nBase carregada com sucesso!")
print(f"Total de registros: {len(df_ons)}")

print("\nColunas:")
print(df_ons.columns.tolist())

print("\nDatas inválidas:", df_ons["data_hora"].isna().sum())
print("Cargas ausentes:", df_ons["val_cargaenergiamwmed"].isna().sum())


Arquivos encontrados: 26

Base carregada com sucesso!
Total de registros: 37989

Colunas:
['id_subsistema', 'nom_subsistema', 'data_hora', 'val_cargaenergiamwmed']

Datas inválidas: 0
Cargas ausentes: 13


In [397]:

# --- FILTRA SUBSISTEMA SUL ---

# Filtrar somente o subsistema Sul
df_ons_sul = df_ons[
    df_ons["id_subsistema"] == "S"
].copy()

# Criar coluna contendo somente a data
df_ons_sul["data"] = df_ons_sul["data_hora"].dt.normalize()

# Remover somente o registro adicional sem carga de 02/02/2013
df_ons_sul = df_ons_sul[
    ~(
        (df_ons_sul["data"] == pd.Timestamp("2013-02-02")) &
        (df_ons_sul["val_cargaenergiamwmed"].isna())
    )
].copy()

# Ordenar cronologicamente
df_ons_sul = df_ons_sul.sort_values("data").reset_index(drop=True)

# Manter somente as colunas necessárias
df_ons_sul = df_ons_sul[
    ["data", "id_subsistema", "nom_subsistema", "val_cargaenergiamwmed"]
]

# Verificar a base preparada
print("Registros do subsistema Sul:", len(df_ons_sul))
print("Datas distintas:", df_ons_sul["data"].nunique())
print("Valores de carga ausentes:", df_ons_sul["val_cargaenergiamwmed"].isna().sum())
print("Datas duplicadas:", df_ons_sul["data"].duplicated().sum())

print("\nPeríodo:")
print("Inicial:", df_ons_sul["data"].min())
print("Final:", df_ons_sul["data"].max())

print("\nRegistros com carga ausente:")
print(
    df_ons_sul[
        df_ons_sul["val_cargaenergiamwmed"].isna()
    ].to_string(index=False)
)

print("\nPrimeiras linhas:")
print(df_ons_sul.head())


Registros do subsistema Sul: 9497
Datas distintas: 9497
Valores de carga ausentes: 3
Datas duplicadas: 0

Período:
Inicial: 2000-01-01 00:00:00
Final: 2025-12-31 00:00:00

Registros com carga ausente:
      data id_subsistema nom_subsistema  val_cargaenergiamwmed
2013-12-01             S            Sul                    NaN
2014-02-01             S            Sul                    NaN
2015-04-09             S            Sul                    NaN

Primeiras linhas:
        data id_subsistema nom_subsistema  val_cargaenergiamwmed
0 2000-01-01             S            Sul            4800.650000
1 2000-01-02             S            Sul            4899.800000
2 2000-01-03             S            Sul            6261.554167
3 2000-01-04             S            Sul            6733.741667
4 2000-01-05             S            Sul            6961.170833


In [398]:

import holidays

# Identificar os anos presentes na base
anos_unicos = sorted(df_ons_sul["data"].dt.year.unique())

# Obter os feriados nacionais brasileiros
feriados_br = holidays.country_holidays(
    "BR",
    years=anos_unicos,
    observed=False,
    language="pt_BR"
)

# Manter somente os feriados nacionais aplicáveis ao período
feriados_nacionais = [
    data
    for data, nome in feriados_br.items()
    if "Sexta-feira Santa" not in nome
    and not (data.month == 11 and data.day == 2 and data.year < 2002)
]

# Identificar feriados nacionais
df_ons_sul["is_feriado"] = (
    df_ons_sul["data"].dt.date.isin(feriados_nacionais).astype(int)
)

# Identificar o dia da semana (0 = segunda-feira; 6 = domingo)
df_ons_sul["dia_semana"] = df_ons_sul["data"].dt.day_of_week

# Identificar finais de semana
df_ons_sul["is_fds"] = (
    df_ons_sul["dia_semana"].isin([5, 6]).astype(int)
)

# Identificar dias úteis (segunda a sexta, exceto feriados)
df_ons_sul["is_dia_util"] = (
    (df_ons_sul["is_feriado"] == 0) &
    (df_ons_sul["is_fds"] == 0)
).astype(int)

# Classificar cada dia
df_ons_sul["tipo_dia"] = "Dia útil"

df_ons_sul.loc[
    df_ons_sul["is_fds"] == 1, "tipo_dia"
] = "Final de semana"

df_ons_sul.loc[
    df_ons_sul["is_feriado"] == 1, "tipo_dia"
] = "Feriado nacional"

# Exibir resultados
print("Distribuição dos tipos de dia:")
print(df_ons_sul["tipo_dia"].value_counts())

print("\nQuantidade de feriados nacionais:", df_ons_sul["is_feriado"].sum())


Distribuição dos tipos de dia:
tipo_dia
Dia útil            6637
Final de semana     2653
Feriado nacional     207
Name: count, dtype: int64

Quantidade de feriados nacionais: 207


In [399]:
# --- LEITURA DOS ARQUIVOS DO INMET ---

from pathlib import Path

pasta_inmet = Path("../dados/INMET")

arquivos_inmet = sorted(pasta_inmet.rglob("*.csv"))

print("Total de arquivos CSV:", len(arquivos_inmet))

Total de arquivos CSV: 1681


In [400]:
# --- IDENTIFICAÇÃO DAS ESTAÇÕES METEOROLÓGICAS ---

import re
import pandas as pd

dados_estacoes = []

padrao = re.compile(
    r"INMET_S_(PR|SC|RS)_(A\d+)_(.*?)_(\d{2}-\d{2}-\d{4})_A_(\d{2}-\d{2}-\d{4})",
    re.IGNORECASE
)

for arquivo in arquivos_inmet:
    resultado = padrao.search(arquivo.stem)

    if resultado:
        estado, codigo, municipio, inicio, fim = resultado.groups()

        dados_estacoes.append({
            "estado": estado.upper(),
            "codigo": codigo.upper(),
            "municipio": municipio.strip(),
            "data_inicio": pd.to_datetime(inicio, format="%d-%m-%Y"),
            "data_fim": pd.to_datetime(fim, format="%d-%m-%Y")
        })

df_estacoes = pd.DataFrame(dados_estacoes)

print("Arquivos identificados:", len(df_estacoes))
print("Estações distintas:", df_estacoes["codigo"].nunique())
print("Estados:", sorted(df_estacoes["estado"].unique()))

Arquivos identificados: 1623
Estações distintas: 92
Estados: ['PR', 'RS', 'SC']


In [401]:
# --- SELEÇÃO DE ESTAÇÕES METEOROLÓGICAS ---

estacoes_selecionadas = [
    # Paraná
    ("PR", "A807"),
    ("PR", "A818"),
    ("PR", "A820"),
    ("PR", "A823"),
    ("PR", "A835"),

    # Santa Catarina
    ("SC", "A806"),
    ("SC", "A817"),
    ("SC", "A841"),
    ("SC", "A845"),
    ("SC", "A851"),

    # Rio Grande do Sul
    ("RS", "A802"),
    ("RS", "A803"),
    ("RS", "A804"),
    ("RS", "A805"),
    ("RS", "A826")
]

df_selecao = df_estacoes[
    df_estacoes.set_index(["estado", "codigo"]).index.isin(
        estacoes_selecionadas
    )
].copy()

display(
    df_selecao[
        ["estado", "codigo", "municipio",
         "data_inicio", "data_fim"]
    ].sort_values(["estado", "codigo"])
)

print("Total de arquivos:", len(df_selecao))

,estado,codigo,municipio,data_inicio,data_fim
11,PR,A807,CURITIBA,2003-01-28,2003-12-31
18,PR,A807,CURITIBA,2004-01-01,2004-12-31
25,PR,A807,CURITIBA,2005-01-01,2005-12-31
32,PR,A807,CURITIBA,2006-01-01,2006-12-31
59,PR,A807,CURITIBA,2007-01-01,2007-12-31
...,...,...,...,...,...
1247,SC,A851,ITAPOA,2021-01-01,2021-12-31
1336,SC,A851,ITAPOA,2022-01-01,2022-12-31
1427,SC,A851,ITAPOA,2023-01-01,2023-12-31
1517,SC,A851,ITAPOA,2024-01-01,2024-12-31


Total de arquivos: 323


In [402]:
# --- FUNÇÕES AUXILIARES PARA LEITURA DOS ARQUIVOS DO INMET ---

import numpy as np
import pandas as pd
import re
import unicodedata

def normalizar_nome(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(c for c in nome if not unicodedata.combining(c))
    return re.sub(r"[^A-Z0-9]", "", nome.upper())


def ler_arquivo_inmet(arquivo):
    df = pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1",
        skiprows=8
    )

    df.columns = df.columns.str.strip()

    colunas_normalizadas = {
        normalizar_nome(coluna): coluna
        for coluna in df.columns
    }

    def encontrar_coluna(inicio):
        inicio = normalizar_nome(inicio)

        for nome, original in colunas_normalizadas.items():
            if nome.startswith(inicio):
                return original

        raise ValueError(
            f"Coluna não encontrada: {inicio} | Arquivo: {arquivo.name}"
        )

    colunas = {
        "data": encontrar_coluna("DATA"),
        "hora_utc": encontrar_coluna("HORA"),
        "precipitacao": encontrar_coluna("PRECIPITACAO TOTAL"),
        "temperatura": encontrar_coluna("TEMPERATURA DO AR BULBO SECO"),
        "umidade": encontrar_coluna("UMIDADE RELATIVA DO AR")
    }

    df = df[list(colunas.values())].rename(
        columns={v: k for k, v in colunas.items()}
    )

    for coluna in ["precipitacao", "temperatura", "umidade"]:
        df[coluna] = pd.to_numeric(
            df[coluna].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ).replace(-9999, np.nan)

    df["data_hora_utc"] = pd.to_datetime(
        df["data"].astype(str) + " " + df["hora_utc"].astype(str),
        errors="coerce",
        utc=True
    )

    df["data_hora_local"] = (
        df["data_hora_utc"].dt.tz_convert("America/Sao_Paulo")
    )

    df["data_local"] = df["data_hora_local"].dt.date

    return df

In [403]:
# --- LEITURA DAS 15 ESTAÇÕES ----

dados_horarios = []

for arquivo in arquivos_inmet:
    resultado = padrao.search(arquivo.stem)

    if resultado:
        estado, codigo, municipio, inicio, fim = resultado.groups()

        if (estado.upper(), codigo.upper()) in estacoes_selecionadas:
            df = ler_arquivo_inmet(arquivo)

            df["estado"] = estado.upper()
            df["codigo_estacao"] = codigo.upper()
            df["municipio"] = municipio.strip()

            dados_horarios.append(df)

df_inmet = pd.concat(dados_horarios, ignore_index=True)

print("Total de registros:", len(df_inmet))
print("Estações:", df_inmet["codigo_estacao"].nunique())
print("Período:", df_inmet["data_local"].min(),
      "a", df_inmet["data_local"].max())

Total de registros: 2746944
Estações: 15
Período: 2001-11-15 a 2025-12-31


In [404]:
# --- AGREGAÇÃO DIÁRIA DAS VARIÁVEIS --- 

fuso = "America/Sao_Paulo"
limite_horas = 0.75

def horas_esperadas(data):
    inicio = pd.Timestamp(data).tz_localize(
        fuso, nonexistent="shift_forward"
    ).tz_convert("UTC")

    fim = (pd.Timestamp(data) + pd.Timedelta(days=1)).tz_localize(
        fuso, nonexistent="shift_forward"
    ).tz_convert("UTC")

    return int((fim - inicio).total_seconds() / 3600)


df_diario_inmet = (
    df_inmet.groupby(
        ["estado", "codigo_estacao", "municipio", "data_local"]
    )
    .agg(
        temperatura=("temperatura", "mean"),
        umidade=("umidade", "mean"),
        precipitacao=("precipitacao", "sum"),
        horas_temperatura=("temperatura", "count"),
        horas_umidade=("umidade", "count"),
        horas_precipitacao=("precipitacao", "count")
    )
    .reset_index()
)

df_diario_inmet["horas_esperadas"] = (
    df_diario_inmet["data_local"].apply(horas_esperadas)
)

for variavel in ["temperatura", "umidade", "precipitacao"]:
    coluna_horas = f"horas_{variavel}"

    df_diario_inmet.loc[
        df_diario_inmet[coluna_horas]
        < df_diario_inmet["horas_esperadas"] * limite_horas,
        variavel
    ] = np.nan
    
print("Registros diários:", len(df_diario_inmet))
print("Estações:", df_diario_inmet["codigo_estacao"].nunique())

Registros diários: 114471
Estações: 15


In [405]:
# --- CONSOLIDAÇÃO CLIMÁTICA REGIONAL ---

df_clima = (
    df_diario_inmet.groupby("data_local")
    .agg(
        temperatura=("temperatura", "mean"),
        umidade=("umidade", "mean"),
        precipitacao=("precipitacao", "mean"),
        estacoes_temperatura=("temperatura", "count"),
        estacoes_umidade=("umidade", "count"),
        estacoes_precipitacao=("precipitacao", "count")
    )
    .reset_index()
)

print("Total de dias:", len(df_clima))
print("Período:", df_clima["data_local"].min(),
      "a", df_clima["data_local"].max())

display(df_clima.head())

Total de dias: 8813
Período: 2001-11-15 a 2025-12-31


,data_local,temperatura,umidade,precipitacao,estacoes_temperatura,estacoes_umidade,estacoes_precipitacao
0,2001-11-15,NaN,NaN,NaN,0,0,0
1,2001-11-16,NaN,NaN,NaN,0,0,0
2,2001-11-17,NaN,NaN,NaN,0,0,0
3,2001-11-18,NaN,NaN,NaN,0,0,0
4,2001-11-19,NaN,NaN,NaN,0,0,0


In [406]:
# --- INTEGRAÇÃO DAS BASES ---

# Padronizar as colunas de data
df_ons_sul["data"] = pd.to_datetime(df_ons_sul["data"])
df_clima["data"] = pd.to_datetime(df_clima["data_local"])

# Integrar novamente os dados do ONS e do INMET

df_base = df_ons_sul.merge(
    df_clima.drop(columns="data_local"),
    on="data",
    how="left",
    validate="one_to_one"
)

print("Registros da base integrada:", len(df_base))
print("Período:", df_base["data"].min(), "a", df_base["data"].max())
print("Colunas:", len(df_base.columns))

Registros da base integrada: 9497
Período: 2000-01-01 00:00:00 a 2025-12-31 00:00:00
Colunas: 15


In [407]:
# --- VERIFICAÇÃO DE VALORES AUSENTES ---

colunas_climaticas = [
    "temperatura",
    "umidade",
    "precipitacao"
]

colunas_verificacao = [
    "val_cargaenergiamwmed"
] + colunas_climaticas

resumo_ausentes = pd.DataFrame({
    "valores_ausentes": df_base[colunas_verificacao].isna().sum(),
    "percentual_ausente": (
        df_base[colunas_verificacao].isna().mean() * 100
    ).round(2)
})

display(resumo_ausentes)

,valores_ausentes,percentual_ausente
val_cargaenergiamwmed,3,0.03
temperatura,760,8.00
umidade,760,8.00
precipitacao,761,8.01


In [408]:
# --- ANÁLISE DE DISPONIBILIDADE CLIMÁTICA ---

clima_completo = df_base[
    ["temperatura", "umidade", "precipitacao"]
].notna().all(axis=1)

clima_ausente = df_base[
    ["temperatura", "umidade", "precipitacao"]
].isna().all(axis=1)

print("Dias com as três variáveis disponíveis:", clima_completo.sum())
print("Dias sem nenhuma variável climática:", clima_ausente.sum())
print("Dias com ausência parcial:", (~clima_completo & ~clima_ausente).sum())

Dias com as três variáveis disponíveis: 8736
Dias sem nenhuma variável climática: 760
Dias com ausência parcial: 1


In [409]:
# Verificar integridade da base integrada

print("Total de registros:", len(df_base))
print("Datas distintas:", df_base["data"].nunique())
print("Datas duplicadas:", df_base["data"].duplicated().sum())

print("\nValores ausentes:")
print(df_base.isna().sum())

print("\nValores suspeitos:")
print("Cargas negativas:", (df_base["val_cargaenergiamwmed"] < 0).sum())
print("Temperaturas ausentes ou fora de -90 a 60 °C:",
      ((df_base["temperatura"] < -90) | (df_base["temperatura"] > 60)).sum())
print("Umidades fora de 0 a 100%:",
      ((df_base["umidade"] < 0) | (df_base["umidade"] > 100)).sum())
print("Precipitações negativas:", (df_base["precipitacao"] < 0).sum())

Total de registros: 9497
Datas distintas: 9497
Datas duplicadas: 0

Valores ausentes:
data                       0
id_subsistema              0
nom_subsistema             0
val_cargaenergiamwmed      3
is_feriado                 0
dia_semana                 0
is_fds                     0
is_dia_util                0
tipo_dia                   0
temperatura              760
umidade                  760
precipitacao             761
estacoes_temperatura     684
estacoes_umidade         684
estacoes_precipitacao    684
dtype: int64

Valores suspeitos:
Cargas negativas: 0
Temperaturas ausentes ou fora de -90 a 60 °C: 0
Umidades fora de 0 a 100%: 0
Precipitações negativas: 0


In [410]:
# Converter tipos de dados antes da exportação

df_base["data"] = pd.to_datetime(df_base["data"])

df_base["is_feriado"] = df_base["is_feriado"].astype(bool)
df_base["is_fds"] = df_base["is_fds"].astype(bool)
df_base["is_dia_util"] = df_base["is_dia_util"].astype(bool)

print(df_base.dtypes)

data                     datetime64[us]
id_subsistema                       str
nom_subsistema                      str
val_cargaenergiamwmed           float64
is_feriado                         bool
dia_semana                        int32
is_fds                             bool
is_dia_util                        bool
tipo_dia                            str
temperatura                     float64
umidade                         float64
precipitacao                    float64
estacoes_temperatura            float64
estacoes_umidade                float64
estacoes_precipitacao           float64
dtype: object


In [411]:
from pathlib import Path

# Criar pasta de destino, caso não exista
pasta_saida = Path("../dados/processados")
pasta_saida.mkdir(parents=True, exist_ok=True)

# Exportar base integrada
caminho_saida = pasta_saida / "ons_projeto_integrado.parquet"

df_base.to_parquet(caminho_saida, index=False)

print("Arquivo Parquet exportado com sucesso!")
print("Caminho:", caminho_saida)

# Exportar base integrada para CSV

caminho_csv = pasta_saida / "ons_projeto_integrado.csv"

df_base.to_csv(
    caminho_csv,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=","
)

print("Arquivo CSV exportado com sucesso!")
print("Caminho:", caminho_csv)

Arquivo Parquet exportado com sucesso!
Caminho: ..\dados\processados\ons_projeto_integrado.parquet
Arquivo CSV exportado com sucesso!
Caminho: ..\dados\processados\ons_projeto_integrado.csv
